In [1]:
import torch.nn as nn
import torch
from torchvision import models
from utils import save_net,load_net
import torch
import torch.nn as nn

# Import the CSRNet model from your model.py file
from model import CSRNet
from model import CSRNet
import torch
import torch.nn as nn

# Import both CSRNet and make_layers from model.py
from model import CSRNet, make_layers



ImportError: bad magic number in 'utils': b'\x03\xf3\r\n'

In [ ]:
class CSRNet(nn.Module):
    def __init__(self, load_weights=False):
        super(CSRNet, self).__init__()
        self.frontend_feat = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512]
        self.backend_feat  = [512, 512, 512,256,128,64]
        self.frontend = make_layers(self.frontend_feat)
        self.backend = make_layers(self.backend_feat,in_channels = 512,dilation = True)
        self.output_layer = nn.Conv2d(64, 1, kernel_size=1)
        if not load_weights:
            mod = models.vgg16(pretrained = True)
            self._initialize_weights()
            for i in range(len(self.frontend.state_dict().items())):
                self.frontend.state_dict().items()[i][1].data[:] = mod.state_dict().items()[i][1].data[:]
    def forward(self,x):
        x = self.frontend(x)
        x = self.backend(x)
        x = self.output_layer(x)
        return x
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

In [ ]:
def make_layers(cfg, in_channels = 3,batch_norm=False,dilation = False):
    if dilation:
        d_rate = 2
    else:
        d_rate = 1
    layers = []
    for v in cfg:
        if v == 'M':
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        else:
            conv2d = nn.Conv2d(in_channels, v, kernel_size=3, padding=d_rate,dilation = d_rate)
            if batch_norm:
                layers += [conv2d, nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
            else:
                layers += [conv2d, nn.ReLU(inplace=True)]
            in_channels = v
    return nn.Sequential(*layers)

In [ ]:
from model import CSRNet
import torch

model = CSRNet()
x = torch.randn(1, 3, 224, 224)
print(model(x).shape)
model = CSRNet()


In [ ]:
x = torch.rand((1,3,255,255))

In [ ]:
model(x).shape

In [ ]:
from model import CSRNet
import torch

# Load CSRNet
model = CSRNet()
model.eval()  # Evaluation mode


In [ ]:
from PIL import Image
import torchvision.transforms as transforms

# Path to your uploaded image
img_path = "croud.jpeg"  # <-- change to your file path

# ImageNet normalization
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load and preprocess
image = Image.open(img_path).convert('RGB')
img_tensor = transform(image).unsqueeze(0)  # shape: [1, 3, H, W]


In [ ]:
from model import CSRNet
import torch
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# 1️⃣ Create model
model = CSRNet()
model.eval()

# 2️⃣ Path to uploaded image
img_path = "croud.jpeg"

# 3️⃣ Preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

image = Image.open(img_path).convert('RGB')
img_tensor = transform(image).unsqueeze(0)  # shape: [1, 3, H, W]

# 4️⃣ Prediction
with torch.no_grad():
    output = model(img_tensor)  # shape: [1, 1, H', W']
    density_map = output.squeeze(0).squeeze(0).numpy()
    count = density_map.sum()

print(f"Estimated Crowd Count: {count:.2f}")

# 5️⃣ Visualization
plt.imshow(density_map, cmap='jet')
plt.title(f"Estimated Count: {count:.2f}")
plt.axis('off')
plt.show()

import matplotlib.pyplot as plt

# Put model in evaluation mode
model.eval()

with torch.no_grad():
    output = model(img_tensor)  # [1, 1, H', W']
    density_map = output.squeeze(0).squeeze(0).numpy()
    count = density_map.sum()

print(f"Estimated Crowd Count: {count:.2f}")

# Visualize
plt.imshow(density_map, cmap='jet')
plt.title(f"Estimated Count: {count:.2f}")
plt.axis('off')
plt.show()
